This notebook estimates domestic production domestically consumed for mineral goods, based on the production data of USGS.

In [1]:
import pandas as pd
import numpy as np
import sqlite3
import brightway2 as bw2
import country_converter as coco

Connect to both SQL databases (BACI with all the data and the filtered version)

In [2]:
conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/trade_data.db')

full_baci_conn = sqlite3.connect('C://Users/11max/PycharmProjects/Regioinvent/trade_data/v5/baci_trade_data_full_version.db')

Read the formatted data from USGS including all the production data. This data was extracted thrugh another notebook available on demand.

In [3]:
prod_minerals = pd.read_excel('USGS_production_data.xlsx', dtype=str)
# change Year and Value to integer and float
prod_minerals['Year'] = prod_minerals['Year'].astype(int)
prod_minerals['Value'] = prod_minerals['Value'].astype(float)
# remove 2025 (no data from BACI for that year)
prod_minerals = prod_minerals.loc[prod_minerals.Year.isin([2020, 2021, 2022, 2023, 2024])]
# remove global total value
prod_minerals = prod_minerals.loc[~prod_minerals.Country.str.contains('[Ww]orld', regex=True)]
# aggregate HS codes columns
prod_minerals = prod_minerals.set_index(['Commodity', 'Statistics_detail', 'Year', 'Value', 'Country']).stack().droplevel(-1).reset_index()
prod_minerals = prod_minerals.rename(columns={0: 'HS codes'})

In [4]:
# extract relevant export values from BACI
required_hs_export_values = set(prod_minerals.loc[:, prod_minerals.columns.str.contains('HS codes')].stack().values.tolist())

placeholders = ', '.join([f"'{str(val)}'" for val in required_hs_export_values])

query = f"SELECT * FROM [International trade data] WHERE cmdCode IN ({placeholders})"
data = pd.read_sql(query, full_baci_conn)

# determine net exports
import_data = data.set_index(['cmdCode', 'refYear', 'importer', 'exporter']).drop(
    ['value (1,000 USD)'], axis=1).groupby(['cmdCode', 'refYear', 'importer']).sum().reset_index()
export_data = data.set_index(['cmdCode', 'refYear', 'exporter']).drop(
    ['value (1,000 USD)', 'importer'], axis=1).groupby(['cmdCode', 'refYear', 'exporter']).sum().reset_index()
import_data = import_data.rename(columns={'importer': 'country', 'quantity (t)': 'import_qty'})
export_data = export_data.rename(columns={'exporter': 'country', 'quantity (t)': 'export_qty'})

# outer merge to ensure we don't lose data 
net_exports_data = pd.merge(
    export_data, 
    import_data, 
    on=['cmdCode', 'refYear', 'country'], 
    how='outer'
)

# fill NaNs with 0 (so subtraction works if one side is missing)
net_exports_data[['export_qty', 'import_qty']] = net_exports_data[['export_qty', 'import_qty']].fillna(0)

# calculate Net Exports
net_exports_data['net_export'] = net_exports_data['export_qty'] - net_exports_data['import_qty']

net_exports_data = net_exports_data[['cmdCode', 'refYear', 'country', 'net_export']]

# remove negative net export balance
net_exports_data = net_exports_data[net_exports_data.loc[:,'net_export'] > 0].dropna()

Similarly to with FAOSTAT data, we use exports to split HS codes to more aggregated USGS commodities. The same specific cases are also treated in the same way here.

In [5]:
# Calculate total global export volume for every HS code
global_hs_totals = net_exports_data.groupby('cmdCode')['net_export'].sum().reset_index()
global_hs_totals.columns = ['HS codes', 'global_export_volume']

for commodity in set(prod_minerals.Commodity):
    for stat in set(prod_minerals.Statistics_detail):
        df = prod_minerals.loc[(prod_minerals.Commodity == commodity) & (prod_minerals.Statistics_detail == stat)]
        if not df.empty:
            codes = set(df.loc[:,'HS codes'])
            if len(codes) > 1:
                # calculate total exports for the commodities
                total_export_cmd = net_exports_data.loc[net_exports_data.cmdCode.isin(codes)].drop('cmdCode', axis=1).groupby(['refYear', 'country']).sum().reset_index()
                total_export_cmd = total_export_cmd.rename(columns={'net_export': 'total_net_export'})
                # filter the net exports data for only the relevant commodities
                net_exports_cmd = net_exports_data.loc[net_exports_data.cmdCode.isin(codes)]
                # calculate the distribution shares of each commodity in each country, based on export shares
                net_exports_cmd = net_exports_cmd.merge(total_export_cmd, on=['refYear', 'country'], how='outer')
                net_exports_cmd.loc[:, 'net_export_share'] = net_exports_cmd.net_export / net_exports_cmd.total_net_export
                # merge this export share data with production data
                dff = df.merge(net_exports_cmd.drop(['net_export', 'total_net_export'], axis=1), 
                               left_on=['HS codes', 'Year', 'Country'], 
                               right_on=['cmdCode', 'refYear', 'country'], how='left').drop(['cmdCode', 'refYear', 'country'], axis=1)
                # fill with 0 if the sum of shares between the different commodities equals to 1
                current_group_sums = dff.groupby(['Commodity', 'Statistics_detail', 'Year', 'Country'])['net_export_share'].transform('sum')
                is_group_complete = np.isclose(current_group_sums, 1.0, atol=1e-3)
                dff['net_export_share'] = np.where(is_group_complete & dff['net_export_share'].isna(), 0.0, dff['net_export_share'])
                # for countries where for a few years no export data was available, but for some other years yes, we apply the average over the studied years
                average_shares = dff.groupby(['Country', 'HS codes'])['net_export_share'].transform('mean')
                dff['net_export_share'] = dff['net_export_share'].fillna(average_shares)
                # for countries where there are no export data available at all, we apply global average export shares between the commodities
                commodity_hs_mapping = dff[['Commodity', 'HS codes']].drop_duplicates()
                global_shares = pd.merge(commodity_hs_mapping, global_hs_totals, on='HS codes', how='left').fillna(0)
                global_commodity_total = global_shares.groupby('Commodity')['global_export_volume'].transform('sum')
                global_shares['global_share'] = np.where(
                    global_commodity_total > 0,
                    global_shares['global_export_volume'] / global_commodity_total,
                    1 / global_shares.groupby('Commodity')['HS codes'].transform('count') # Hard fallback if code never appears globally
                )
                # Create a clean dictionary/mapping for quick lookup
                global_share_map = global_shares.set_index(['Commodity', 'HS codes'])['global_share'].to_dict()
                remaining_nans = dff['net_export_share'].isna()
                if remaining_nans.any():
                    nan_pairs = dff.loc[remaining_nans, ['Commodity', 'HS codes']].to_records(index=False)
                    dff.loc[remaining_nans, 'net_export_share'] = [global_share_map.get(tuple(pair), 0) for pair in nan_pairs]
                # apply the net export ratios to production values to split between the commodities
                dff.loc[:, 'Value'] *= dff.loc[:, 'net_export_share']
                dff = dff.drop('net_export_share', axis=1)
                # replace old values with the new ones
                prod_minerals = prod_minerals.drop(prod_minerals.loc[(prod_minerals.Commodity == commodity) & (prod_minerals.Statistics_detail == stat)].index)
                prod_minerals = pd.concat([prod_minerals, dff])

USGS provides data in tonnes of metal content for some commodities while BACI only references gross weight. Thus, we convert the relevant USGS data to gross weight, through average metal content estimates obtained through Gemini3.5 flash.

In [6]:
metal_contents = {
    '720221': 0.75,
    '720229': 0.45,
    '260200': 0.42,
    '261390': 0.50,
    '261310': 0.57,
    '260400': 0.10,
    '310420': 0.60,
    '310430': 0.51,
    '261610': 0.035,
    '280700': 0.32,
    '260900': 0.60,
    '260800': 0.50,
    '260300': 0.25,
    '260500': 0.10,
    '260700': 0.55
}

# specific case for HS commodity 261590
metal_content_261590 = {
    'Niobium': 0.65,
    'Tantalum': 0.25,
    'Vanadium': 0.56
}

for metal in metal_contents.keys():
    prod_minerals.loc[prod_minerals.loc[:,'HS codes'] == metal, 'Value'] /= metal_contents[metal]

for metal in metal_content_261590.keys():
    prod_minerals.loc[(prod_minerals.loc[:,'HS codes'] == '261590') & (prod_minerals.Commodity == metal)]

In [7]:
# merge to then subtract net exports from production value
prod_minerals = prod_minerals.merge(net_exports_data, left_on=['HS codes', 'Year', 'Country'], right_on=['cmdCode', 'refYear', 'country'], how='left').loc[:, ['Year', 'HS codes', 'Value', 'net_export', 'Country']]
prod_minerals.loc[:, 'Value'] = prod_minerals.loc[:, 'Value'] - prod_minerals.loc[:, 'net_export'].fillna(0)

In [8]:
prod_minerals = prod_minerals.drop('net_export', axis=1)
prod_minerals = prod_minerals[prod_minerals.Value > 0].fillna(0)

In the USGS data, there are some "Other countries" region. Instead of excluding those, we apply a similar logic. We look at export market shares of countries within the year for the commodity. We then apply these shares to countries that form this "Other countries" region.

In [9]:
for cmd in set(prod_minerals.loc[:,'HS codes']):
    for year in set(prod_minerals.Year):
        df = prod_minerals.loc[(prod_minerals.loc[:,'HS codes'] == cmd) & (prod_minerals.loc[:,'Year'] == year)]
        covered_countries = df.loc[~df.Country.str.contains('Other'), 'Country'].tolist()
        other_countries_net_exports = net_exports_data.loc[(net_exports_data.loc[:,'cmdCode'] == cmd) & 
                                                            (net_exports_data.loc[:,'refYear'] == year) & 
                                                            ~(net_exports_data.country.isin(covered_countries))]
        other_countries_net_exports.loc[:, 'net_export'] /= other_countries_net_exports.net_export.sum()
        # if IndexError -> there is no "other" region in prod data
        try:
            other_countries_net_exports.loc[:, 'net_export'] *= df.loc[df.Country.str.contains('Other'), 'Value'].iloc[0]
        except IndexError:
            continue
        other_countries_net_exports = other_countries_net_exports.rename(columns={'cmdCode': 'HS codes', 'refYear':'Year', 'country': 'Country', 'net_export':'Value'})
        prod_minerals = prod_minerals.drop(df.index)
        df = df.drop(df.loc[df.Country.str.contains('Other')].index)
        df = pd.concat([df, other_countries_net_exports])
        prod_minerals = pd.concat([prod_minerals, df])

In [10]:
# formatting
prod_minerals = prod_minerals.rename(columns={'Country': 'producer', 'Year': 'refYear', 'HS codes': 'cmdCode', 'Value': 'quantity (t)'})
prod_minerals.loc[:,'source'] = 'Estimated from USGS MCS commodity files.'
prod_minerals.loc[:,'country_coverage'] = 'complete'

In [ ]:
# USGS relies on some regions that are not aligned with the UN COMTRADE regions, we fix that
prod_minerals.loc[prod_minerals.producer == 'GLP', 'producer'] = 'FRA'
prod_minerals.loc[prod_minerals.producer == 'TWN', 'producer'] = 'S19' # the "Other Asia, nes" region is used to represent a proxy of Taiwan trade in UN COMTRADE

Add the data to the SQL database inside the "Real production" data table